In [1]:
import pandas as pd

# Load the processed dataset
df = pd.read_csv("../data/processed/news_dataset.csv")

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 3 rows:")
print(df.head(3))

Dataset shape: (44689, 2)

Columns:
['content', 'label']

First 3 rows:
                                             content  label
0  Merkel, Juncker discuss Catalan crisis: EU off...      1
1  LIBERAL HACK KATIE COURIC Says Fake News Is “T...      0
2  TRUMP WAS RIGHT! Audit Reveals State Dept. Cyb...      0


In [2]:
# Inspect a few raw articles before preprocessing

for i in range(3):
    print(f"\n--- Article {i+1} ---")
    print("Label:", df.loc[i, "label"])
    print("Text:")
    print(df.loc[i, "content"][:1000])


--- Article 1 ---
Label: 1
Text:
Merkel, Juncker discuss Catalan crisis: EU official BRUSSELS (Reuters) - German Chancellor Angela Merkel and the European Commission President Jean-Claude Juncker have discussed the Catalan crisis in a phone call, an EU official told Reuters on Saturday, in a sign of growing unease over potential disruption across Europe. The call signals fresh concern over a possible declaration of independence next week by the Catalan regional parliament and the reaction of the Spanish government, which could exacerbate what is already Spain s worst political crisis for decades. Juncker spoke to Merkel from India on Friday on the sidelines of an EU-India summit.  They discussed the situation in Spain, among other things,  the EU official said, without adding details. Concern is growing in EU capitals about the negative impact of the crisis on the Spanish economy, the fourth largest in the euro zone, and on possible spillovers to other economies of the bloc.  European

In [3]:
import re

def clean_text(text):
    # Convert to lowercase
    text = text.lower()
    
    # Remove URLs
    text = re.sub(r"http\S+|www\S+|https\S+", "", text)
    
    # Remove HTML tags
    text = re.sub(r"<.*?>", "", text)
    
    # Remove punctuation and special characters
    text = re.sub(r"[^a-zA-Z\s]", " ", text)
    
    # Remove extra whitespace
    text = re.sub(r"\s+", " ", text).strip()
    
    return text

In [4]:
sample_text = df.loc[0, "content"]

cleaned_sample = clean_text(sample_text)

print("BEFORE:")
print(sample_text[:500])

print("\nAFTER:")
print(cleaned_sample[:500])

BEFORE:
Merkel, Juncker discuss Catalan crisis: EU official BRUSSELS (Reuters) - German Chancellor Angela Merkel and the European Commission President Jean-Claude Juncker have discussed the Catalan crisis in a phone call, an EU official told Reuters on Saturday, in a sign of growing unease over potential disruption across Europe. The call signals fresh concern over a possible declaration of independence next week by the Catalan regional parliament and the reaction of the Spanish government, which could 

AFTER:
merkel juncker discuss catalan crisis eu official brussels reuters german chancellor angela merkel and the european commission president jean claude juncker have discussed the catalan crisis in a phone call an eu official told reuters on saturday in a sign of growing unease over potential disruption across europe the call signals fresh concern over a possible declaration of independence next week by the catalan regional parliament and the reaction of the spanish government which

In [6]:
# Apply text cleaning to the complete dataset
df["cleaned_content"] = df["content"].apply(clean_text)

print("Cleaning completed!")

print("\nOriginal:")
print(df.loc[0, "content"][:300])

print("\nCleaned:")
print(df.loc[0, "cleaned_content"][:300])

Cleaning completed!

Original:
Merkel, Juncker discuss Catalan crisis: EU official BRUSSELS (Reuters) - German Chancellor Angela Merkel and the European Commission President Jean-Claude Juncker have discussed the Catalan crisis in a phone call, an EU official told Reuters on Saturday, in a sign of growing unease over potential di

Cleaned:
merkel juncker discuss catalan crisis eu official brussels reuters german chancellor angela merkel and the european commission president jean claude juncker have discussed the catalan crisis in a phone call an eu official told reuters on saturday in a sign of growing unease over potential disruption


In [7]:
# Check for empty text after cleaning
empty_cleaned = (df["cleaned_content"].str.len() == 0).sum()

print("Empty articles after cleaning:", empty_cleaned)

# Show a few if any exist
if empty_cleaned > 0:
    print("\nExamples:")
    print(df.loc[df["cleaned_content"].str.len() == 0, ["content", "label"]].head())
    

Empty articles after cleaning: 9

Examples:
                                                 content  label
2339   https://100percentfedup.com/12-yr-old-black-co...      0
12112  https://fedup.wpengine.com/wp-content/uploads/...      0
14811  https://fedup.wpengine.com/wp-content/uploads/...      0
16147  https://fedup.wpengine.com/wp-content/uploads/...      0
23048  https://100percentfedup.com/video-hillary-aske...      0


In [8]:
# Remove articles with no usable text after cleaning
before = len(df)

df = df[df["cleaned_content"].str.len() > 0].reset_index(drop=True)

after = len(df)

print("Articles before removal:", before)
print("Empty articles removed:", before - after)
print("Articles remaining:", after)

Articles before removal: 44689
Empty articles removed: 9
Articles remaining: 44680


In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Create TF-IDF vectorizer
tfidf = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    max_features=50000
)

print("TF-IDF vectorizer created successfully!")



TF-IDF vectorizer created successfully!


In [11]:
# Fit TF-IDF on the cleaned news articles
X_tfidf = tfidf.fit_transform(df["cleaned_content"])

print("TF-IDF conversion completed!")

print("Number of articles:", X_tfidf.shape[0])
print("Number of features:", X_tfidf.shape[1])
print("TF-IDF matrix shape:", X_tfidf.shape)


TF-IDF conversion completed!
Number of articles: 44680
Number of features: 50000
TF-IDF matrix shape: (44680, 50000)


In [12]:
# Inspect the first 5 articles and first 10 TF-IDF features
print(X_tfidf[:5, :10].toarray())

[[0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]]


In [13]:
# Show the first 20 learned TF-IDF features
print(tfidf.get_feature_names_out()[:20])

['aapl' 'aaron' 'abadi' 'abadi said' 'abandon' 'abandon the' 'abandoned'
 'abandoning' 'abbas' 'abbott' 'abc' 'abc news' 'abc this' 'abcpolitics'
 'abd' 'abdel' 'abdel fattah' 'abducted' 'abduction' 'abdul']


In [14]:
# Get TF-IDF features and values for the first article
feature_names = tfidf.get_feature_names_out()

first_article = X_tfidf[0]

nonzero_indices = first_article.nonzero()[1]
values = first_article.data

for index, value in zip(nonzero_indices[:20], values[:20]):
    print(f"{feature_names[index]:30} {value:.4f}")

merkel                         0.1225
juncker                        0.1627
discuss                        0.0660
catalan                        0.2410
crisis                         0.1204
eu                             0.3208
official                       0.0749
brussels                       0.0800
reuters                        0.0280
german                         0.0367
chancellor                     0.0405
angela                         0.0407
and                            0.0431
european                       0.0615
commission                     0.0693
president                      0.0132
jean                           0.0474
claude                         0.0533
have                           0.0113
discussed                      0.0729


In [15]:
from sklearn.model_selection import train_test_split

# Input text
X = df["cleaned_content"]

# Target labels
y = df["label"]

# Split into training and testing data
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

print("\nTraining label distribution:")
print(y_train.value_counts())

print("\nTesting label distribution:")
print(y_test.value_counts())

Training samples: 35744
Testing samples: 8936

Training label distribution:
label
0    18775
1    16969
Name: count, dtype: int64

Testing label distribution:
label
0    4694
1    4242
Name: count, dtype: int64


In [16]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Create a fresh TF-IDF vectorizer
tfidf = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    max_features=50000
)

# Learn vocabulary ONLY from training data
X_train_tfidf = tfidf.fit_transform(X_train)

# Transform test data using the same vocabulary
X_test_tfidf = tfidf.transform(X_test)

print("TF-IDF completed correctly!")

print("Training TF-IDF shape:", X_train_tfidf.shape)
print("Testing TF-IDF shape:", X_test_tfidf.shape)


TF-IDF completed correctly!
Training TF-IDF shape: (35744, 50000)
Testing TF-IDF shape: (8936, 50000)
